# 벡터데이터 기반 RAG 어플리케이션 실습

# 🚀 목표는 뭘까?

**RAG (Retrieval-Augmented Generation)** 시스템을 만들어보는 실습입니다.  
즉, **웹에서 정보를 가져오고 → 벡터로 기억한 뒤 → GPT가 그 정보를 활용해 답변**하게 만드는 AI 챗봇을 만들어보는 거예요.

---

## 📦 전체 흐름 요약

### 1단계: 웹 페이지에서 정보 가져오기  
> 마치 구글에서 웹문서를 긁어오듯, 원하는 웹 문서를 자동으로 수집합니다.

### 2단계: 문서를 잘게 나누기 (텍스트 분할)  
> 문서가 너무 길면 AI가 이해하기 어려워요.  
> 그래서 일정 길이(예: 1000자)로 잘라서 슬라이스합니다.

### 3단계: 벡터 임베딩 생성 & 저장  
> 잘린 문장을 **벡터(숫자 리스트)**로 바꿔 저장합니다.  
> 이 과정을 통해 문장을 AI가 기억할 수 있어요.

### 4단계: 질문이 들어오면 관련 문장 검색  
> 사용자의 질문과 가장 관련 있는 벡터(문장)를 찾아옵니다.

### 5단계: 질문 + 관련 문장 → GPT에게 전달  
> “이 문장들을 참고해서 이 질문에 답해줘!” 하고 GPT에게 넘겨줍니다.

### 6단계: GPT가 최종 답변 생성  
> GPT가 관련 정보를 바탕으로 짧고 정확하게 답변합니다.

---

## 🎯 쉽게 비유하자면...

- 🔍 **웹 문서**는 책장에 꽂힌 책이고  
- ✂️ **텍스트 분할**은 책을 챕터별로 자르는 작업이고  
- 🧠 **벡터화**는 챕터를 AI가 기억할 수 있게 만드는 일이고  
- ❓ **질문**은 “이 책에서 ~~내용 알려줘”라는 요청이고  
- 🤖 **GPT**는 책을 읽고 정확한 답을 찾아주는 똑똑한 사서입니다.

---

## 💡 왜 RAG를 쓸까요?

단순히 GPT에게 질문하면 **모르거나 헛소리할 가능성**이 있어요.  
RAG는 GPT가 **실제 자료를 참고**해서 답하게 도와줍니다.  
즉, **GPT를 지식 기반 AI로 한 단계 업그레이드**시키는 방법입니다!


In [ ]:
# 필요한 외부 라이브러리를 설치합니다.
# 포인트: 이 라이브러리들은 GPT와 외부 데이터를 연결하고 처리하는 데 꼭 필요합니다.
!pip install --upgrade openai langchain langchain-openai langchain-community beautifulsoup4 langchainhub -q
!pip install chromadb tiktoken -q

In [ ]:
# openai 모듈과 os 모듈을 불러옵니다. (os는 환경 변수 설정에 필요)
import openai
import os
# OpenAI API 키를 환경 변수로 등록합니다.
# 포인트: 이렇게 하면 코드에 키가 노출되지 않고 안전하게 사용할 수 있습니다.
os.environ['OPENAI_API_KEY']=""
# OpenAI API와 연결합니다.
client = openai.OpenAI()
print("API 키 설정이 완료되었습니다.")


API 키 설정이 완료되었습니다.


In [ ]:
# LangChain과 BeautifulSoup 관련 라이브러리 불러오기
import os
from langchain.text_splitter import RecursiveCharacterTextSplitter  # 문서 자를 때 사용
from langchain_community.document_loaders import WebBaseLoader  # 웹 문서 로드
from langchain_community.vectorstores import Chroma  # 벡터 DB 저장소
from langchain_core.output_parsers import StrOutputParser  # 출력 형식 변환기
from langchain_core.runnables import RunnablePassthrough  # 입력 그대로 넘겨주는 모듈
from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # GPT 모델 및 임베딩 생성
import bs4

# 웹에서 문서를 가져오는 WebBaseLoader를 설정합니다.
# 포인트: 지정한 웹 URL에서 특정 HTML 클래스에 해당하는 본문 내용만 추출하도록 합니다.
loader = WebBaseLoader(
    web_paths=("https://mlrx.tistory.com/entry/%EC%B1%97%EB%B4%87-PDF-QA-%EC%B1%97%EB%B4%87-%EA%B0%9C%EB%B0%9C%ED%95%98%EA%B8%B0-1",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(class_="article_view")  # 본문 텍스트만 추출
    ),
)
docs = loader.load()


In [ ]:
docs  # 웹에서 가져온 문서 객체 확인

print(docs)  # 문서 내용 출력
print(type(docs))  # 문서의 데이터 타입 확인 (문서 분할 전에 데이터 형태 점검)

[Document(metadata={'source': 'https://mlrx.tistory.com/entry/%EC%B1%97%EB%B4%87-PDF-QA-%EC%B1%97%EB%B4%87-%EA%B0%9C%EB%B0%9C%ED%95%98%EA%B8%B0-1'}, page_content='\n\n\n목적\nPDF 문서 내용을 기반으로 질의응답(QA)를 할 수 있는 인트라넷에서 사용가능한 챗봇 개발\n\xa0\n준비물\npython\nlangchain\nopenai api key\n\xa0\n과정\n전체적인 플로우\n\n\n1. PDF 에서 텍스트 추출하기\nlangchain에서 제공하는 pdf loader를 이용해 pdf에서 text를 추출한다.\nlangchain에서는 다양한 방법을 제공하므로 각자 상황에 맞는 방법을 사용하도록 한다.\n(현재 글쓴이도 적절한 방법을 모색하고 있다.)\nfrom langchain.document_loaders import UnstructuredPDFLoader\nfrom langchain.text_splitter import RecursiveCharacterTextSplitter\n\nloader = UnstructuredPDFLoader(\'../약관.pdf\')\ndata = loader.load()\n\nprint(f"{len(data)}개의 문서를 가지고 있습니다.")\nprint(f"문서에 {len(data[0].page_content)}개의 단어를 가지고 있습니다.")\n1개의 문서를 가지고 있습니다.\n문서에 281012개의 단어를 가지고 있습니다.\n\xa0\n2. 텍스트를 chunk로 나누기\nembedding을 만들기 위해선 embedding이 모델이 소화할 수 있는 양만큼의 입력만 넣어줘야 한다. 그러기에 281012개의 단어를 가진 값을 통채로 넣어줄 수 없으므로 더 작은 단위로 묶어서 넣어주자.\ntext_splitter = RecursiveCharacterTextSplitter(chunk_size=

## 📚 RAG에서 `chunk_size`와 `chunk_overlap`의 역할과 임베딩 검색 성능과의 관계

RAG 시스템에서는 문서를 자르고 벡터화하여 검색하기 때문에,  
**`chunk_size`(조각 크기)와 `chunk_overlap`(조각 간 겹침)** 설정이 임베딩 검색 성능에 직접적인 영향을 미칩니다.

---

### ✅ 1. 임베딩 검색이란?

- 문장을 **벡터(숫자)**로 변환해 저장하고
- 질문이 들어오면 **유사한 벡터를 검색**해 관련 문장을 찾는 방식입니다.
- → 따라서 **가장 적절한 "문서 조각(chunk)"을 얼마나 잘 찾느냐가 핵심**입니다.

---

### ✅ 2. `chunk_size`가 검색 정확도에 미치는 영향

| `chunk_size` 너무 작으면 | `chunk_size` 너무 크면 |
|--------------------------|------------------------|
| 🔹 문맥이 잘림, 내용이 부족함<br>🔹 중복 벡터 많아져 검색 노이즈 증가 | 🔹 문서 여러 주제가 섞임<br>🔹 임베딩이 흐릿해져 질문과 유사도 계산이 부정확함 |

➡️ 적절한 크기(보통 500~1000자)는 **의미 있는 문맥 단위로 자르면서도 내용이 분산되지 않게 합니다.**

---

### ✅ 3. `chunk_overlap`은 검색 재현율(recall)을 높임

- 중요한 단어나 문장이 **chunk 경계에 걸쳐 잘릴 수 있음**
- overlap을 주면 **중요 내용이 인접한 chunk에도 포함**되어,
  질문과 더 잘 매칭될 확률이 높아집니다

🧠 예시:  
질문: *"챗봇 개발 조건은?"*  
→ "개발 조건"이 경계에 걸쳐 있을 경우, **겹침이 없으면 검색 실패**  
→ 겹침이 있으면 **다음 chunk에도 포함되어 정확한 검색 가능**

---

### ✅ 핵심 요약

> **chunk는 AI가 검색할 정보 단위, overlap은 중요한 문맥이 누락되지 않도록 하는 보호막입니다.**  
> **이 두 값의 적절한 조정은 임베딩 기반 검색의 정확도(precision)와 재현율(recall)을 높이는 핵심 요소입니다.**

---

### 💡 참

- `chunk_size ↓`: semantic sparsity (의미 희박)  
- `chunk_size ↑`: semantic drift (의미 혼합, 벡터 불명확)  
- `chunk_overlap`: 문맥 연속성과 정보 손실 방지 역할


In [ ]:
# ✅ 2단계: 문서 텍스트 분할
# 긴 문서를 일정 크기로 잘라주는 작업입니다. GPT가 한 번에 처리할 수 있는 분량으로 조각냅니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# 실제로 문서를 분할합니다. -> splits 에는 잘린 문장 블록들이 담깁니다.
splits = text_splitter.split_documents(docs)
splits


In [ ]:
splits

[Document(metadata={'source': 'https://mlrx.tistory.com/entry/%EC%B1%97%EB%B4%87-PDF-QA-%EC%B1%97%EB%B4%87-%EA%B0%9C%EB%B0%9C%ED%95%98%EA%B8%B0-1'}, page_content="목적\nPDF 문서 내용을 기반으로 질의응답(QA)를 할 수 있는 인트라넷에서 사용가능한 챗봇 개발\n\xa0\n준비물\npython\nlangchain\nopenai api key\n\xa0\n과정\n전체적인 플로우\n\n\n1. PDF 에서 텍스트 추출하기\nlangchain에서 제공하는 pdf loader를 이용해 pdf에서 text를 추출한다.\nlangchain에서는 다양한 방법을 제공하므로 각자 상황에 맞는 방법을 사용하도록 한다.\n(현재 글쓴이도 적절한 방법을 모색하고 있다.)\nfrom langchain.document_loaders import UnstructuredPDFLoader\nfrom langchain.text_splitter import RecursiveCharacterTextSplitter\n\nloader = UnstructuredPDFLoader('../약관.pdf')\ndata = loader.load()"),
 Document(metadata={'source': 'https://mlrx.tistory.com/entry/%EC%B1%97%EB%B4%87-PDF-QA-%EC%B1%97%EB%B4%87-%EA%B0%9C%EB%B0%9C%ED%95%98%EA%B8%B0-1'}, page_content='print(f"{len(data)}개의 문서를 가지고 있습니다.")\nprint(f"문서에 {len(data[0].page_content)}개의 단어를 가지고 있습니다.")\n1개의 문서를 가지고 있습니다.\n문서에 281012개의 단어를 가지고 있습니다.\n\xa0\n2. 텍스트를 chunk로 나누기\nembedding을 만들기 위해선 embe

In [ ]:
# ✅ 3단계: 텍스트를 임베딩(숫자 벡터)으로 변환하고 벡터 DB에 저장
# 포인트: GPT가 문장을 이해할 수 있도록 숫자로 바꾸고, 저장소에 기억시키는 단계입니다.
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

# 저장된 벡터로부터 검색 기능을 만드는 단계
retriever = vectorstore.as_retriever()


In [ ]:
# ✅ 4단계: 질문을 던지고, 관련된 문서를 검색
documents = retriever.get_relevant_documents("챗봇을 개발하기 위한 조건은?")

# 검색된 문서 출력
for document in documents:
    print(f"문서 내용: {document.page_content}")


/tmp/ipython-input-8-3581895282.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  documents = retriever.get_relevant_documents("챗봇을 개발하기 위한 조건은?")


문서 내용: 위 문제를 하나하나 해결해나가며 글을 작성하도록 하겠다.
특히 인트라넷에서 사용 가능해야하므로 openai의 의존성은 완전히 없애서 완성도를 높여야 할 것이다. 또한, 비상업용 모델은 사용할 수 없으므로 상업적으로 사용가능한 dolly 로 대체할 것이다.
 
 
참고
https://www.youtube.com/watch?v=2xxziIWmaSA 
https://www.youtube.com/watch?v=h0DHDp1FbmQ 
https://www.youtube.com/watch?v=ih9PBGVVOO4 




공유하기

게시글 관리


천천히찬찬히
문서 내용: 목적
PDF 문서 내용을 기반으로 질의응답(QA)를 할 수 있는 인트라넷에서 사용가능한 챗봇 개발
 
준비물
python
langchain
openai api key
 
과정
전체적인 플로우


1. PDF 에서 텍스트 추출하기
langchain에서 제공하는 pdf loader를 이용해 pdf에서 text를 추출한다.
langchain에서는 다양한 방법을 제공하므로 각자 상황에 맞는 방법을 사용하도록 한다.
(현재 글쓴이도 적절한 방법을 모색하고 있다.)
from langchain.document_loaders import UnstructuredPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = UnstructuredPDFLoader('../약관.pdf')
data = loader.load()
문서 내용: '대인배상 와 대물배상에서는 보상하지 않는 손해로, 보험계약자나 기명피보험자의 고의로 인한 손해와 기명피보험자 이외의 피보험자의 고의로 인한 손해가 있다.',
 'There is no information provided about damages that are not covered by insurance.',
 "I'm sorry, I cannot provide a final answ

In [ ]:
# ✅ 5단계: 기본 프롬프트 템플릿 정의
# GPT가 응답을 만들 때 따라야 할 지침을 정해줍니다.
from langchain.prompts import PromptTemplate
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "당신은 질문에 답하는 작업을 수행하는 보조자입니다. "
        "다음에 제공된 문맥을 사용하여 질문에 답변하십시오. "
        "답을 모르면 모른다고 말하십시오. "
        "세 문장 이내로 답변하고 간결하게 작성하세요.\n"
        "질문: {question}\n"
        "문맥: {context}\n"
        "답변:"
    ),
)


In [ ]:

# ✅ 6단계: Hub에서 고급 프롬프트 불러오기 (선택사항)
# langchain hub에서 더 정교한 프롬프트를 불러옵니다.
# https://smith.langchain.com/hub/rlm/rag-prompt
# # https://smith.langchain.com/hub 에서 원하는 prompt를 받아올 수 있습니다.

from langchain import hub

# 프롬프트를 가져오기 위해 API 키 설정
os.environ["LANGSMITH_API_KEY"] = "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"

# 고급 프롬프트 가져오기 (더 정교한 지시를 포함)
prompt = hub.pull("rlm/rag-prompt")

In [ ]:
# 프롬프트 내용 출력
print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


In [ ]:
# ✅ 7단계: LLM 설정 및 체인 구성
# 포인트: 질문과 문맥을 GPT에 함께 전달하고, 응답을 받는 전체 처리 흐름을 구성합니다.
llm = ChatOpenAI(model_name="gpt-4o", temperature=0.1)

# 후처리 함수: 여러 문서를 하나의 문자열로 묶는 역할
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)


In [ ]:
# 체인 구성
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}  # 문맥을 검색하고 질문은 그대로 전달
    | prompt  # 문맥 + 질문 → GPT로 전달될 형태로 포맷
    | llm  # GPT가 응답 생성
    | StrOutputParser()  # 출력된 응답을 사람이 읽을 수 있는 텍스트로 정리
)

# ✅ 8단계: 질문에 대해 응답 생성
response = rag_chain.invoke("비상업용 모델은 사용할 수 없으므로 상업적으로 사용가능한 00 로 대체할 것이다.에서 00은? ")
print(response)  # GPT가 생성한 응답 출력

00은 "dolly"입니다.


## RAG 평가지표(고전)

# 📊 주요 평가 지표 정의

### ✅ 1. BLEU (문장 구조 유사도)
- 생성된 문장이 기준 문장의 **단어 배열과 어순**을 얼마나 정확하게 모방했는지를 측정합니다.
- 📌 예: "나는 사과를 먹었다" vs "나는 먹었다 사과를" → 단어는 같지만 어순이 다르면 낮은 점수

### ✅ 2. ROUGE-L (핵심 어구 포함률)
- 기준 문장과 생성 문장에서 **가장 긴 공통 부분 문자열(Longest Common Subsequence)**을 기반으로, 핵심 단어나 구절이 얼마나 일치하는지를 평가합니다.
- 📌 예: "사과를 먹었다" vs "어제 밤에 사과를 먹었다" → 핵심 표현이 포함되어 있어 높은 점수

### ✅ 3. BERTScore (의미적 유사도)
- 문장 전체를 BERT 임베딩 공간에 매핑한 후, **의미적으로 얼마나 유사한지**를 정량화합니다.  
  단어의 형태보다는 **의미론적 일치**에 중점을 둡니다.
- 📌 예: "그녀는 사과를 먹었다" ≈ "여자가 과일을 섭취했다" → 표현은 달라도 의미가 유사해 높은 점수 가능

### ✅ 4. UniEval (표면 단어 일치율)
- 기준 문장과 생성 문장에서 **중복된 단어의 비율**을 단순히 계산합니다.  
  구조나 의미보다는 **공통 단어의 출현 여부**에 집중합니다.
- 📌 예: 기준: "dolly 입니다" / 생성: "이것은 dolly 입니다" → 많은 단어가 겹쳐 높은 점수


In [ ]:
!pip install rouge_score bert_score

In [ ]:
# ✅ 평가 지표 관련 라이브러리 로딩
import re
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
from bert_score import BERTScorer
import torch

# ✅ 전처리 함수 (BERTScore용)
def preprocess_text(text):
    return re.sub(r'[^a-zA-Z\s]', '', text)

# ✅ BERTScore 계산
def calculate_bertscore(generated, reference, model_type='bert-base-uncased'):
    try:
        generated_clean = preprocess_text(generated)
        reference_clean = preprocess_text(reference)
        scorer = BERTScorer(model_type=model_type, lang='en', rescale_with_baseline=True)
        P, R, F1 = scorer.score([generated_clean], [reference_clean])
        return round(F1.mean().item() * 100, 2)
    except Exception as e:
        print(f"BERTScore 오류: {e}")
        return "N/A"

# ✅ BLEU 점수 계산
def calculate_bleu(generated, reference):
    try:
        reference_tokens = [reference.split()]
        candidate_tokens = generated.split()
        return round(sentence_bleu(reference_tokens, candidate_tokens) * 100, 2)
    except Exception as e:
        print(f"BLEU 오류: {e}")
        return "N/A"

# ✅ ROUGE-L 점수 계산
def calculate_rouge(generated, reference):
    try:
        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        score = scorer.score(reference, generated)['rougeL'].fmeasure
        return round(score * 100, 2)
    except Exception as e:
        print(f"ROUGE 오류: {e}")
        return "N/A"

# ✅ UniEval 점수 계산 (공통 단어 기반 단순 유사도)
def calculate_unieval(generated, reference):
    try:
        gen_words = set(generated.split())
        ref_words = set(reference.split())
        if not ref_words:
            return 0.0
        score = len(gen_words & ref_words) / len(ref_words)
        return round(score * 100, 2)
    except Exception as e:
        print(f"UniEval 오류: {e}")
        return "N/A"

# ✅ 기준 정답 예시 (필요 시 변경)
reference_answer = '"dolly"입니다.'

# ✅ 평가 실행
print("\n--- 평가 지표 결과 ---")
print(f"1. BLEU Score: {calculate_bleu(response, reference_answer)}")
print(f"2. ROUGE-L Score: {calculate_rouge(response, reference_answer)}")
print(f"3. BERTScore: {calculate_bertscore(response, reference_answer)}")
print(f"4. UniEval Score: {calculate_unieval(response, reference_answer)}")



--- 평가 지표 결과 ---
1. BLEU Score: 0.0
2. ROUGE-L Score: 66.67


/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

3. BERTScore: 100.0
4. UniEval Score: 100.0


##RAG평가지표 최신

# 🤖 LLM 응답 평가 지표: 개념 쉽게 이해하기

### ✅ 1. Document Relevance (문서 관련성)
- 질문에 비해 정답이 **얼마나 관련 있는 정보만 담고 있는지** 평가
- 📌 예: 질문이 “사과 효능”인데 정답이 “배 효능” 위주면 낮은 점수

### ✅ 2. Answer Faithfulness (사실 충실도)
- 생성된 답이 제시된 사실에 **얼마나 충실하게 기반**했는지 평가
- 📌 예: 사실에는 “A는 B다”라고 했는데 답에 “A는 C다”라고 쓰면 감점

### ✅ 3. Answer Helpfulness (도움 정도)
- 답변이 질문에 대해 **얼마나 유용하고 실질적인 도움**을 주는지 평가
- 📌 예: 질문에 대해 “잘 모르겠어요” 같은 응답은 낮은 점수

### ✅ 4. Answer Correctness (정답 정확도)
- 생성된 답이 기준 정답과 **얼마나 정확히 일치하는지** 평가
- 📌 예: 기준 답이 “파리는 프랑스 수도”인데 “파리는 이탈리아 수도”면 낮은 점수


In [ ]:
import re
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

# LLM 초기화
llm_eval = ChatOpenAI(model_name="gpt-4o", temperature=0.1)
parser = StrOutputParser()

# LLM 평가 체인 생성 함수
def build_llm_chain(template_text):
    prompt = PromptTemplate.from_template(template_text)
    return (
        {"question": RunnablePassthrough(), "model_output": RunnablePassthrough(), "original_text": RunnablePassthrough()}
        | prompt
        | llm_eval
        | parser
    )

# 템플릿들
doc_rel_template = """You are a grader. Grade how relevant the FACTS are to the QUESTION. Return only a score from 0 to 100.\nQUESTION: {question}\nFACTS: {original_text}"""
faithfulness_template = """You are a grader. Grade how well the STUDENT ANSWER is grounded in the FACTS. Return only a score from 0 to 100.\nFACTS: {original_text}\nSTUDENT ANSWER: {model_output}"""
helpfulness_template = """You are a grader. Grade how helpful the STUDENT ANSWER is in answering the QUESTION. Return only a score from 0 to 100.\nQUESTION: {question}\nSTUDENT ANSWER: {model_output}"""
correctness_template = """You are a grader. Grade how factually correct the STUDENT ANSWER is compared to the GROUND TRUTH. Return only a score from 0 to 100.\nQUESTION: {question}\nGROUND TRUTH: {original_text}\nSTUDENT ANSWER: {model_output}"""

# 체인 구성
chain_doc_rel = build_llm_chain(doc_rel_template)
chain_faithfulness = build_llm_chain(faithfulness_template)
chain_helpfulness = build_llm_chain(helpfulness_template)
chain_correctness = build_llm_chain(correctness_template)

# 점수 추출 함수
def score_from_chain(chain, model_output, reference_answer, question):
    try:
        output = chain.invoke({
            "question": question,
            "model_output": model_output,
            "original_text": reference_answer
        })
        match = re.search(r'\d+', output)
        return int(match.group()) if match else "N/A"
    except Exception as e:
        print(f"평가 오류: {e}")
        return "N/A"

# ✅ 사용자 입력 받기
question = input("질문을 입력하세요:\n")
reference_answer = input("정답(Reference Answer)을 입력하세요:\n")
response = input("GPT 응답을 입력하세요:\n")

# ✅ 평가 실행
print("\n--- LLM 기반 평가 지표 ---")
print(f"1. Document Relevance: {score_from_chain(chain_doc_rel, response, reference_answer, question)}")
print(f"2. Answer Faithfulness: {score_from_chain(chain_faithfulness, response, reference_answer, question)}")
print(f"3. Answer Helpfulness: {score_from_chain(chain_helpfulness, response, reference_answer, question)}")
print(f"4. Answer Correctness: {score_from_chain(chain_correctness, response, reference_answer, question)}")


질문을 입력하세요:
비상업용 모델은 사용할 수 없으므로 상업적으로 사용가능한 00 로 대체할 것이다.에서 00은? 
정답(Reference Answer)을 입력하세요:
"dolly"입니다.
GPT 응답을 입력하세요:
00은 "dolly"입니다.

--- LLM 기반 평가 지표 ---
1. Document Relevance: 100
2. Answer Faithfulness: 100
3. Answer Helpfulness: 100
4. Answer Correctness: 100
